In [ ]:
import numpy as np
from sklearn.metrics import f1_score, classification_report
import pandas as pd
df = pd.read_csv("eval_subset_no_readme_cleanup_only_cos_sim.csv", encoding="utf-8-sig")

ylabels = [f"sdg{i}" for i in range(1,18)]


y_true = df[[f"act_sdg{i}" for i in range(1,18)]].values
y_pred = df[[f"pred_readme_sdg{i}" for i in range(1,18)]].values

# Convert predictions to float for label_ranking_average_precision_score
y_score = type(y_pred)  # Check type of first element
import numpy as np

y_pred_numeric = np.array(y_pred[0:], dtype=float)
y_score =  (y_pred_numeric > 0.4).astype(int)

# print(len(y_score[0]))
# print(len(y_true[0]))

y_pred_numeric.shape, y_true.shape

((117, 17), (117, 17))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

df = pd.read_csv("sdg_pred_readme_noreadmecleanup.csv", encoding="utf-8-sig")

y_true = df[[f"act_sdg{i}" for i in range(1, 18)]].values.astype(int)
y_prob = df[[f"pred_readme_sdg{i}" for i in range(1, 18)]].values.astype(float)
y_pred = (y_prob > 0.4).astype(int)

selected = set()

# 1 — minimum 2 per SDG category
for i in range(17):
    positives = np.where(y_true[:, i] == 1)[0]
    if len(positives) > 0:
        chosen = np.random.choice(positives, size=min(2, len(positives)), replace=False)
        selected.update(chosen.tolist())

# 2 — at least 8 multi-label projects
multilabel = np.where(y_true.sum(axis=1) > 1)[0]
selected.update(
    np.random.choice(multilabel, size=min(8, len(multilabel)), replace=False).tolist()
)

# 3 — at least 10 hard cases (lowest F1 per project)
f1_per_project = np.array([
    f1_score(y_true[i], y_pred[i], average='macro', zero_division=0)
    for i in range(len(df))
])
hard = np.where(f1_per_project < np.percentile(f1_per_project, 30))[0]
selected.update(
    np.random.choice(hard, size=min(10, len(hard)), replace=False).tolist()
)

# 4 — fill to 35 randomly
remaining = 35 - len(selected)
if remaining > 0:
    unselected = [i for i in range(len(df)) if i not in selected]
    selected.update(
        np.random.choice(unselected, size=min(remaining, len(unselected)), replace=False).tolist()
    )

indices = list(selected)[:35]

# Save subset as CSV — permanent, never regenerate
subset_df = df.iloc[indices].reset_index(drop=True)
subset_df.to_csv('eval_subset.csv', index=False)

print(f"Subset size: {len(subset_df)}")
print(f"SDG support in subset: {y_true[indices].sum(axis=0)}")
print(f"Multi-label projects: {(y_true[indices].sum(axis=1) > 1).sum()}")
print("Saved to eval_subset.csv")

Subset size: 35
SDG support in subset: [ 9  7 12 12  4  4  2  7  8 10  8  4  6  2  3 10  8]
Multi-label projects: 24
Saved to eval_subset.csv


: 

In [26]:
df.head()

,name,github_url,project_description,act_sdg1,act_sdg2,act_sdg3,act_sdg4,act_sdg5,act_sdg6,act_sdg7,act_sdg8,act_sdg9,act_sdg10,act_sdg11,act_sdg12,act_sdg13,act_sdg14,act_sdg15,act_sdg16,act_sdg17
0,Aam Digital,https://github.com/Aam-Digital/ndb-core,Easy-to-use case management software for the s...,1,0,1,1,1,0,0,1,0,1,0,0,0,0,0,1,0
1,Accessible Kazakhstan,https://github.com/qlt2020/doskaz,It is a model of an online map with informatio...,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,Accessible Medical Records via Integrated Tech...,https://github.com/PSMRI/AMRIT,AMRIT is an open-source platform enhancing pri...,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0
3,AccessMod,https://github.com/unige-geohealth/accessmod,"AccessMod is a free, open-source, standalone s...",0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,African Storybook,https://github.com/global-asp/global-asp,Provides open access to picture storybooks in ...,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [2]:
df["act_sdg17"].to_list()

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0]

In [7]:
import numpy as np
from sklearn.metrics import label_ranking_average_precision_score

# True binary labels (samples x labels)
y_true = np.array(y_true)
# Model prediction scores (must be continuous)
y_score = np.array(y_pred)

# Calculate label ranking average precision
lrap = label_ranking_average_precision_score(y_true, y_score)

print("Label Ranking Average Precision:", lrap)




Label Ranking Average Precision: 0.4141968870485837


In [12]:
per_category_f1 = f1_score(y_true, y_score, average=None, zero_division=0)
macro_f1 = f1_score(y_true, y_score, average='macro', zero_division=0)

SDG_LABELS = [f"pred_sdg{i}" for i in range(1, 18)]

weighted_f1 = f1_score(y_true, y_score, average='weighted', zero_division=0)

print(classification_report(
    y_true,
    y_score,
    target_names=SDG_LABELS,
    zero_division=0
))


# print(classification_report(
#     y_true,
#     y_pred,
#     target_names=SDG_LABELS,
#     zero_division=0
# ))



print(f"Macro F1:    {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

# --- Step 7: Per category breakdown as dataframe ---
category_results = pd.DataFrame({
    'SDG': SDG_LABELS,
    'F1_Score': per_category_f1,
    'Support': y_true.sum(axis=0).astype(int)  # how many positive examples per SDG
}).sort_values('F1_Score', ascending=False)

print("Per Category F1:")
print(category_results.to_string(index=False))

              precision    recall  f1-score   support

   pred_sdg1       0.13      0.87      0.23        15
   pred_sdg2       0.09      0.82      0.17        11
   pred_sdg3       0.38      0.93      0.53        42
   pred_sdg4       0.24      0.84      0.37        25
   pred_sdg5       0.12      0.83      0.22        12
   pred_sdg6       0.08      1.00      0.14         8
   pred_sdg7       0.03      1.00      0.07         3
   pred_sdg8       0.15      0.73      0.25        15
   pred_sdg9       0.31      0.88      0.46        34
  pred_sdg10       0.10      0.41      0.16        17
  pred_sdg11       0.19      0.56      0.28        16
  pred_sdg12       0.15      0.75      0.26         8
  pred_sdg13       0.19      0.87      0.31        15
  pred_sdg14       0.03      0.50      0.06         2
  pred_sdg15       0.10      0.80      0.17         5
  pred_sdg16       0.31      0.80      0.45        30
  pred_sdg17       0.17      0.65      0.27        20

   micro avg       0.17   

# checking for sdgs with less projects 1. sdg-6, 8 projects, no bart, testing weak v/s concentrated descriptions 

In [1]:
sdg_proj_6 = df[df["act_sdg6"] == '1'][["name", "project_description"]]
sdg_proj_6.to_dict(orient="records")

NameError: name 'df' is not defined

these are the predictions for pred sdg 6 , readme 

SDG_DESCS = ["""SDG 1 : extreme poverty income below dollar threshold cash transfers 
    social protection floors universal basic coverage poorest vulnerable 
    populations economic resources land property inheritance microfinance 
    disaster risk resilience eradicate destitution livelihood safety nets
    proportion population poverty line consumption expenditure""",

    """SDG2 : hunger food insecurity malnutrition stunting wasting underweight 
    smallholder farmers agricultural productivity crop yields soil fertility 
    seed banks genetic diversity food systems rural markets price volatility 
    famine emergency food aid nutrition programmes dietary diversity 
    sustainable agriculture irrigation agroecology""",

    """SDG 3 : health maternal mortality neonatal child under-five deaths communicable disease 
    HIV AIDS tuberculosis malaria hepatitis vaccines immunization epidemic 
    non-communicable cardiovascular cancer diabetes mental health substance abuse 
    universal health coverage medicines essential drugs reproductive health 
    family planning skilled birth attendance""",

    """SDG 4 : quality education primary secondary school completion literacy numeracy dropout rates 
    early childhood education vocational training technical skills TVET 
    higher education scholarships qualified teachers learning outcomes 
    disability inclusive education gender parity enrolment attendance 
    digital literacy foundational skills""",

    """SDG 5 : gender equality women empowerment discrimination violence against women 
    female genital mutilation child marriage unpaid domestic care work 
    equal pay wage gap sexual reproductive rights contraception 
    women leadership political participation land ownership property rights 
    girls education menstrual hygiene""",

    """SDG 6 : drinking water safe sanitation open defecation WASH hygiene handwashing 
    water quality treatment wastewater recycling water use efficiency 
    water scarcity transboundary river basin groundwater aquifer 
    water infrastructure rural piped supply irrigation systems 
    water borne disease cholera dysentery""",

    """SDG 7 : electricity access clean cooking fuels renewable energy solar wind 
    hydropower geothermal biomass energy efficiency appliances buildings 
    fossil fuel subsidy reform grid infrastructure off grid mini grid 
    energy poverty kerosene candles kilowatt megawatt gigawatt 
    carbon emission energy transition""",

    """SDG 8 : decent work employment job creation unemployment youth labour 
    forced labour modern slavery human trafficking child labour 
    informal economy living wage productivity GDP growth 
    small medium enterprises entrepreneurship financial inclusion 
    banking credit microfinance tourism sustainable business""",

    """SDG 9 : infrastructure roads bridges ports railways broadband internet 
    connectivity ICT access rural urban industrialization manufacturing 
    value added industry research development innovation patents 
    technology transfer mobile network 4G 5G fintech 
    resilient infrastructure disaster proof construction""",

    """SDG 10 :income inequality gini coefficient wealth distribution top bottom 
    decile palma ratio social mobility discrimination race ethnicity 
    disability migrants remittances immigration policy 
    developing country representation voting rights 
    progressive taxation redistribution fiscal policy""",

    """SDG 11 : urban cities slums informal settlements affordable housing 
    public transport sustainable mobility pedestrian cycling 
    air pollution particulate matter urban planning zoning 
    cultural heritage disaster risk reduction flood resilience 
    green spaces parks waste collection municipal solid waste""",

    """SDG 12 : sustainable consumption production lifecycle footprint 
    toxic chemicals hazardous waste electronic waste e-waste 
    circular economy recycling repair reuse reduce 
    food waste loss supply chain corporate sustainability reporting 
    consumer awareness green procurement public spending""",

    """SDG 13 : climate change greenhouse gas emissions carbon dioxide methane 
    global warming temperature rise adaptation mitigation 
    climate resilience extreme weather drought flood cyclone 
    Paris agreement nationally determined contributions NDC 
    climate finance loss damage early warning systems""",

    """SDG 14 : ocean marine coastal fisheries overfishing illegal unreported 
    coral reef seagrass mangrove wetland biodiversity 
    plastic pollution marine debris ocean acidification 
    deep sea mining small island developing states 
    exclusive economic zone aquaculture blue economy""",

    """SDG 15 : terrestrial forest deforestation land degradation desertification 
    biodiversity species extinction poaching trafficking wildlife 
    protected areas national park conservation habitat restoration 
    invasive species mountain ecosystem dryland 
    land tenure soil carbon sequestration reforestation""",

    """SDG 16 : peace conflict violence armed groups rule of law justice 
    access to legal aid courts accountable institutions 
    corruption bribery transparency freedom of information 
    press freedom civil society participation human rights 
    birth registration identity documents stateless persons 
    illicit financial flows money laundering tax evasion""",

    """SDG 17 : global partnership development finance ODA official aid 
    debt relief developing countries technology transfer capacity building 
    trade WTO multilateral system policy coherence 
    data statistics monitoring reporting voluntary national review 
    south south cooperation multi stakeholder blended finance"""
]

AND THE SDG DESCS AS SDG LABLES

In [ ]:
url_sdg_6_preds_concentrated_sdg_descs = [{'project_name': 'WorldHealthOrganization/godata', 'project_url': 'https://github.com/WorldHealthOrganization/godata', 'sdg_predictions': {'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.911, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.899, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.898, 'SDG 5: Achieve gender equality and empower all women and girls': 0.867, 'SDG 1: End poverty in all its forms everywhere': 0.864, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.82, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.811, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.769, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.765, 'SDG 4: Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all': 0.758}},

{'project_name': 'Zenysis/Harmony', 'project_url': 'https://github.com/Zenysis/Harmony', 'sdg_predictions': {'SDG 1: End poverty in all its forms everywhere': 0.824, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.823, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.777, 'SDG 5: Achieve gender equality and empower all women and girls': 0.777, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.763, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.725, 'SDG 4: Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all': 0.662, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.643, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.594, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.57}},

{'project_name': 'unicef/hope', 'project_url': 'https://github.com/unicef/hope', 'sdg_predictions': {'SDG 1: End poverty in all its forms everywhere': 0.879, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.856, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.797, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.79, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.779, 'SDG 4: Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all': 0.755, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.754, 'SDG 5: Achieve gender equality and empower all women and girls': 0.737, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.731, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.722}},

{'project_name': 'hotosm/tasking-manager', 'project_url': 'https://github.com/hotosm/tasking-manager/', 'sdg_predictions': {'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.93, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.891, 'SDG 1: End poverty in all its forms everywhere': 0.891, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.886, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.837, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.823, 'SDG 5: Achieve gender equality and empower all women and girls': 0.809, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.806, 'SDG 12: Ensure sustainable consumption and production patterns': 0.803, 'SDG 10: Reduce inequality within and among countries': 0.798}},

{'project_name': 'opengisch/QField', 'project_url': 'https://github.com/opengisch/QField', 'sdg_predictions': {'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.844, 'SDG 1: End poverty in all its forms everywhere': 0.836, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.744, 'SDG 4: Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all': 0.658, 'SDG 5: Achieve gender equality and empower all women and girls': 0.638, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.583, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.566, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.55, 'SDG 10: Reduce inequality within and among countries': 0.501, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.485}},

{'project_name': 'Rural-Environmental-Registry/core', 'project_url': 'https://github.com/Rural-Environmental-Registry/core', 'sdg_predictions': {'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.877, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.868, 'SDG 1: End poverty in all its forms everywhere': 0.858, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.799, 'SDG 5: Achieve gender equality and empower all women and girls': 0.781, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.777, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.761, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.758, 'SDG 4: Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all': 0.737, 'SDG 10: Reduce inequality within and among countries': 0.67}},

{'project_name': 'ushahidi/platform', 'project_url': 'https://github.com/ushahidi/platform/', 'sdg_predictions': {'SDG 1: End poverty in all its forms everywhere': 0.641, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.638, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.578, 'SDG 4: Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all': 0.566, 'SDG 5: Achieve gender equality and empower all women and girls': 0.537, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.492, 'SDG 10: Reduce inequality within and among countries': 0.469, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.4, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.394, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.372}},

{'project_name': 'upyog/UPYOG', 'project_url': 'https://github.com/upyog/UPYOG', 'sdg_predictions': {'SDG 1: End poverty in all its forms everywhere': 0.547, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.531, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.49, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.476, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.45, 'SDG 5: Achieve gender equality and empower all women and girls': 0.422, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.414, 'SDG 4: Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all': 0.376, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.372, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.342}}]

In [ ]:
for item in url_sdg_6_preds_concentrated_sdg_descs:
    for sdg, score in item["sdg_predictions"].items():
        if sdg.startswith("SDG 6:"):
            print(f"{item['project_name']} - {sdg}: {score}")

WorldHealthOrganization/godata - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.898
Zenysis/Harmony - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.777
unicef/hope - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.797
hotosm/tasking-manager - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.806
opengisch/QField - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.55
Rural-Environmental-Registry/core - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.758
ushahidi/platform - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.578
upyog/UPYOG - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.45


NOW USE 

SDG_DESCS = [
    'Goal 1 calls for an end to poverty in all its manifestations, including extreme poverty, over the next 15 years. All people everywhere, including the poorest and most vulnerable, should enjoy a basic standard of living and social protection benefits.',

    'Goal 2 seeks to end hunger and all forms of malnutrition and to achieve sustainable food production by 2030. It is premised on the idea that everyone should have access to sufficient nutritious food, which will require widespread promotion of sustainable agriculture, a doubling of agricultural productivity, increased investments and properly functioning food markets.',

    'Goal 3 aims to ensure health and well-being for all at all ages by improving reproductive, maternal and child health; ending the epidemics of major communicable diseases; reducing non-communicable and environmental diseases; achieving universal health coverage; and ensuring access to safe, affordable and effective medicines and vaccines for all.',

    'Goal 4 focuses on the acquisition of foundational and higher-order skills; greater and more equitable access to technical and vocational education and training and higher education; training throughout life; and the knowledge, skills and values needed to function well and contribute to society.',

    'Goal 5 aims to empower women and girls to reach their full potential, which requires eliminating all forms of discrimination and violence against them, including harmful practices. It seeks to ensure that they have every opportunity for sexual and reproductive health and reproductive rights; receive due recognition for their unpaid work; have full access to productive resources; and enjoy equal participation with men in political, economic and public life.',

    'Goal 6 goes beyond drinking water, sanitation and hygiene to also address the quality and sustainability of water resources. Achieving this Goal, which is critical to the survival of people and the planet, means expanding international cooperation and garnering the support of local communities in improving water and sanitation management.',

    'Goal 7 seeks to promote broader energy access and increased use of renewable energy, including through enhanced international cooperation and expanded infrastructure and technology for clean energy.',

    'Goal 8 aims to provide opportunities for full and productive employment and decent work for all while eradicating forced labour, human trafficking and child labour.',

    'Goal 9 focuses on the promotion of infrastructure development, industrialization and innovation. This can be accomplished through enhanced international and domestic financial, technological and technical support, research and innovation, and increased access to information and communication technology.',

    'Goal 10 calls for reducing inequalities in income, as well as those based on sex, age, disability, race, class, ethnicity, religion and opportunity—both within and among countries. It also aims to ensure safe, orderly and regular migration and addresses issues related to representation of developing countries in global decision-making and development assistance.',

    'Goal 11 aims to renew and plan cities and other human settlements in a way that fosters community cohesion and personal security while stimulating innovation and employment.',

    'Goal 12 aims to promote sustainable consumption and production patterns through measures such as specific policies and international agreements on the management of materials that are toxic to the environment.',

    'Climate change presents the single biggest threat to development, and its widespread, unprecedented effects disproportionately burden the poorest and the most vulnerable. Urgent action is needed not only to combat climate change and its impacts, but also to build resilience in responding to climate-related hazards and natural disasters.',

    'Goal 14 seeks to promote the conservation and sustainable use of marine and coastal ecosystems, prevent marine pollution and increase the economic benefits to small island developing States and LDCs from the sustainable use of marine resources.',

    'Goal 15 focuses on managing forests sustainably, restoring degraded lands and successfully combating desertification, reducing degraded natural habitats and ending biodiversity loss. All of these efforts in combination will help ensure that livelihoods are preserved for those that depend directly on forests and other ecosystems, that biodiversity will thrive, and that the benefits of these natural resources will be enjoyed for generations to come.',

    'Goal 16 envisages peaceful and inclusive societies based on respect for human rights, the rule of law, good governance at all levels, and transparent, effective and accountable institutions. Many countries still face protracted violence and armed conflict, and far too many people are poorly supported by weak institutions and lack access to justice, information and other fundamental freedoms.',

    'The 2030 Agenda requires a revitalized and enhanced global partnership that mobilizes all available resources from Governments, civil society, the private sector, the United Nations system and other actors. Increasing support to developing countries, in particular LDCs, landlocked developing countries and small island developing States is fundamental to equitable progress for all.'
]

USING THE PREVIOUS DESCS AS LABELS, now doing it




In [ ]:
url_sdg_6_preds_not_concentrated_sdg_descs=[{'project_name': 'WorldHealthOrganization/godata', 'project_url': 'https://github.com/WorldHealthOrganization/godata', 'sdg_predictions': {'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.953, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.877, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.859, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.852, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.832, 'SDG 1: End poverty in all its forms everywhere': 0.831, 'SDG 5: Achieve gender equality and empower all women and girls': 0.81, 'SDG 4: Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all': 0.75, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.734, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.732}},
{'project_name': 'Zenysis/Harmony', 'project_url': 'https://github.com/Zenysis/Harmony', 'sdg_predictions': {'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.787, 'SDG 1: End poverty in all its forms everywhere': 0.776, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.758, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.758, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.735, 'SDG 4: Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all': 0.73, 'SDG 5: Achieve gender equality and empower all women and girls': 0.709, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.621, 'SDG 10: Reduce inequality within and among countries': 0.562, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.552}},
{'project_name': 'unicef/hope', 'project_url': 'https://github.com/unicef/hope', 'sdg_predictions': {'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.915, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.881, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.866, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.863, 'SDG 4: Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all': 0.86, 'SDG 1: End poverty in all its forms everywhere': 0.837, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.826, 'SDG 10: Reduce inequality within and among countries': 0.813, 'SDG 5: Achieve gender equality and empower all women and girls': 0.802, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.786}},
{'project_name': 'hotosm/tasking-manager', 'project_url': 'https://github.com/hotosm/tasking-manager/', 'sdg_predictions': {'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.93, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.912, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.901, 'SDG 4: Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all': 0.89, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.86, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.859, 'SDG 10: Reduce inequality within and among countries': 0.839, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.834, 'SDG 5: Achieve gender equality and empower all women and girls': 0.821, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.815}},
{'project_name': 'opengisch/QField', 'project_url': 'https://github.com/opengisch/QField', 'sdg_predictions': {'SDG 1: End poverty in all its forms everywhere': 0.81, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.797, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.762, 'SDG 4: Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all': 0.737, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.603, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.602, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.561, 'SDG 5: Achieve gender equality and empower all women and girls': 0.539, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.509, 'SDG 10: Reduce inequality within and among countries': 0.409}},
{'project_name': 'ushahidi/platform', 'project_url': 'https://github.com/ushahidi/platform/', 'sdg_predictions': {'SDG 1: End poverty in all its forms everywhere': 0.53, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.524, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.511, 'SDG 4: Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all': 0.466, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.464, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.443, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.434, 'SDG 5: Achieve gender equality and empower all women and girls': 0.428, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.422, 'SDG 15: Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss': 0.372}},
{'project_name': 'upyog/UPYOG', 'project_url': 'https://github.com/upyog/UPYOG', 'sdg_predictions': {'SDG 1: End poverty in all its forms everywhere': 0.512, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.506, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.49, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.476, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.452, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.448, 'SDG 5: Achieve gender equality and empower all women and girls': 0.429, 'SDG 4: Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all': 0.391, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.387, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.375}}
]

In [ ]:
for item in url_sdg_6_preds_not_concentrated_sdg_descs:
    for sdg, score in item["sdg_predictions"].items():
        if sdg.startswith("SDG 6:"):
            print(f"{item['project_name']} - {sdg}: {score}")

WorldHealthOrganization/godata - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.877
Zenysis/Harmony - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.758
unicef/hope - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.863
hotosm/tasking-manager - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.86
opengisch/QField - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.561
ushahidi/platform - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.464
upyog/UPYOG - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.448


In [ ]:
project_description_preds_sdg_6 = [{'project_name': 'Go.Data', 'project_url': '', 'sdg_predictions': {'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 1.0, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.782, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.757, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.736, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.736, 'SDG 17: Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development': 0.729, 'SDG 16: Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels': 0.729, 'SDG 15: Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss': 0.646, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.591, 'SDG 5: Achieve gender equality and empower all women and girls': 0.549}, 'method': 'ensemble', 'text_length': 261},
{'project_name': 'Harmony', 'project_url': '', 'sdg_predictions': {'SDG 16: Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels': 1.0, 'SDG 17: Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development': 0.979, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.952, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.879, 'SDG 5: Achieve gender equality and empower all women and girls': 0.844, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.781, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.715, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.696, 'SDG 10: Reduce inequality within and among countries': 0.672, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.493}, 'method': 'ensemble', 'text_length': 391},
{'project_name': 'HOPE', 'project_url': '', 'sdg_predictions': {'SDG 17: Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development': 1.0, 'SDG 1: End poverty in all its forms everywhere': 0.978, 'SDG 16: Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels': 0.876, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.851, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.84, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.719, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.663, 'SDG 12: Ensure sustainable consumption and production patterns': 0.587, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.552, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.507}, 'method': 'ensemble', 'text_length': 277},
{'project_name': 'HOT Tasking Manager', 'project_url': '', 'sdg_predictions': {'SDG 1: End poverty in all its forms everywhere': 1.0, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.905, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.839, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.832, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.746, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.721, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.707, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.635, 'SDG 5: Achieve gender equality and empower all women and girls': 0.571, 'SDG 12: Ensure sustainable consumption and production patterns': 0.369}, 'method': 'ensemble', 'text_length': 263},
{'project_name': 'QField', 'project_url': '', 'sdg_predictions': {'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 1.0, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.629, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.55, 'SDG 10: Reduce inequality within and among countries': 0.541, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.458, 'SDG 17: Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development': 0.323, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.31, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.273, 'SDG 15: Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss': 0.262, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.248}, 'method': 'ensemble', 'text_length': 217},
{'project_name': 'Rural Environmental Registry Registration Module', 'project_url': '', 'sdg_predictions': {'SDG 15: Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss': 1.0, 'SDG 12: Ensure sustainable consumption and production patterns': 0.839, 'SDG 14: Conserve and sustainably use the oceans, seas and marine resources for sustainable development': 0.806, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.79, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.766, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.733, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.591, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.497, 'SDG 1: End poverty in all its forms everywhere': 0.487, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.368}, 'method': 'ensemble', 'text_length': 216},
{'project_name': 'The Ushahidi Platform', 'project_url': '', 'sdg_predictions': {'SDG 1: End poverty in all its forms everywhere': 1.0, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.999, 'SDG 15: Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss': 0.857, 'SDG 16: Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels': 0.826, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.754, 'SDG 5: Achieve gender equality and empower all women and girls': 0.703, 'SDG 10: Reduce inequality within and among countries': 0.693, 'SDG 17: Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development': 0.603, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.592, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.534}, 'method': 'ensemble', 'text_length': 261},
{'project_name': 'UPYOG', 'project_url': '', 'sdg_predictions': {'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 1.0, 'SDG 17: Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development': 0.864, 'SDG 16: Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels': 0.821, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.771, 'SDG 1: End poverty in all its forms everywhere': 0.766, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.644, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.601, 'SDG 15: Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss': 0.554, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.518, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.434}, 'method': 'ensemble', 'text_length': 506}]

In [ ]:
import pandas as pd

df = pd.read_csv("sdg_pred_readme_noreadmecleanup.csv", encoding="utf-8-sig")
df.shape

df.head()

,name,github_url,project_description,act_sdg1,act_sdg2,act_sdg3,act_sdg4,act_sdg5,act_sdg6,act_sdg7,...,pred_readme_sdg8,pred_readme_sdg9,pred_readme_sdg10,pred_readme_sdg11,pred_readme_sdg12,pred_readme_sdg13,pred_readme_sdg14,pred_readme_sdg15,pred_readme_sdg16,pred_readme_sdg17
0,Aam Digital,https://github.com/Aam-Digital/ndb-core,Easy-to-use case management software for the s...,1,0,1,1,1,0,0,...,0.667,0.710,0.000,0.609,0.000,0.0,0.0,0.0,0.0,0.0
1,Accessible Kazakhstan,https://github.com/qlt2020/doskaz,It is a model of an online map with informatio...,1,0,0,0,0,0,0,...,0.000,0.000,0.000,0.000,0.304,0.0,0.0,0.0,0.0,0.0
2,Accessible Medical Records via Integrated Tech...,https://github.com/PSMRI/AMRIT,AMRIT is an open-source platform enhancing pri...,0,0,1,0,0,0,0,...,0.814,0.871,0.834,0.000,0.000,0.0,0.0,0.0,0.0,0.0
3,AccessMod,https://github.com/unige-geohealth/accessmod,"AccessMod is a free, open-source, standalone s...",0,0,1,0,0,0,0,...,0.689,0.686,0.740,0.659,0.000,0.0,0.0,0.0,0.0,0.0
4,Aid Management Platform,https://github.com/devgateway/amp,AMP helps governments and development partners...,0,0,0,0,0,0,0,...,0.000,0.589,0.000,0.000,0.000,0.0,0.0,0.0,0.0,0.0


: 

clean readme sdg 6 projects


In [5]:
preds_clean_readme = [{'project_name': 'WorldHealthOrganization/godata', 'project_url': 'https://github.com/WorldHealthOrganization/godata', 'sdg_predictions': {'SDG 17: Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development': 1.0, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.668, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.574, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.566, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.515, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.496, 'SDG 15: Protect, restore and promote sustainable useof terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss': 0.431, 'SDG 12: Ensure sustainable consumption and production patterns': 0.386, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.357, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.336}},
{'project_name': 'Zenysis/Harmony', 'project_url': 'https://github.com/Zenysis/Harmony', 'sdg_predictions': {'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 1.0, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.874, 'SDG 5: Achieve gender equality and empower all women and girls': 0.646, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.599, 'SDG 17: Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development': 0.582, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.539, 'SDG 16: Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels': 0.531, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.496, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.405, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.272}},
{'project_name': 'unicef/hope', 'project_url': 'https://github.com/unicef/hope', 'sdg_predictions': {'SDG 17: Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development': 1.0, 'SDG 1: End poverty in all its forms everywhere': 0.897, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.655, 'SDG 16: Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels': 0.655, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.646, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.574, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.546, 'SDG 12: Ensure sustainable consumption and production patterns': 0.473, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.439, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.417}},
{'project_name': 'hotosm/tasking-manager', 'project_url': 'https://github.com/hotosm/tasking-manager/', 'sdg_predictions': {'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 1.0, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.845, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.746, 'SDG 17: Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development': 0.734, 'SDG 1: End poverty in all its forms everywhere': 0.706, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.601, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.509, 'SDG 12: Ensure sustainable consumption and production patterns': 0.444, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.304, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.301}},
{'project_name': 'opengisch/QField', 'project_url': 'https://github.com/opengisch/QField', 'sdg_predictions': {'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 1.0, 'SDG 17: Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development': 0.811, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.781, 'SDG 12: Ensure sustainable consumption and production patterns': 0.758, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.697, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.676, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.659, 'SDG 10: Reduce inequality within and among countries': 0.587, 'SDG 16: Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels': 0.556, 'SDG 15: Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss': 0.506}},
{'project_name': 'Rural-Environmental-Registry/core', 'project_url': 'https://github.com/Rural-Environmental-Registry/core', 'sdg_predictions': {'SDG 15: Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss': 1.0, 'SDG 13: Take urgent action to combat climate change and its impacts': 0.84, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.82, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.79, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.79, 'SDG 1: End poverty in all its forms everywhere': 0.745, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.72, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.575, 'SDG 14: Conserve and sustainably use the oceans, seas and marine resources for sustainable development': 0.498, 'SDG 12: Ensure sustainable consumption and production patterns': 0.49}},
{'project_name': 'ushahidi/platform', 'project_url': 'https://github.com/ushahidi/platform/', 'sdg_predictions': {'SDG 14: Conserve and sustainably use the oceans, seas and marine resources for sustainable development': 1.0, 'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 0.904, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.899, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.866, 'SDG 1: End poverty in all its forms everywhere': 0.808, 'SDG 10: Reduce inequality within and among countries': 0.772, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.734, 'SDG 12: Ensure sustainable consumption and production patterns': 0.707, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.636, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.51}},
{'project_name': 'upyog/UPYOG', 'project_url': 'https://github.com/upyog/UPYOG', 'sdg_predictions': {'SDG 9: Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation': 1.0, 'SDG 2: End hunger, achieve food security and improved nutrition and promote sustainable agriculture': 0.566, 'SDG 11: Make cities and human settlements inclusive, safe, resilient and sustainable': 0.555, 'SDG 1: End poverty in all its forms everywhere': 0.459, 'SDG 17: Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development': 0.424, 'SDG 6: Ensure availability and sustainable management of water and sanitation for all': 0.399, 'SDG 3: Ensure healthy lives and promote well-being for all at all ages': 0.396, 'SDG 7: Ensure access to affordable, reliable, sustainable and modern energy for all': 0.372, 'SDG 16: Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels': 0.279, 'SDG 8: Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all': 0.274}}]



In [6]:
for item in preds_clean_readme:
    for sdg, score in item["sdg_predictions"].items():
        if sdg.startswith("SDG 6:"):
            print(f"{item['project_name']} - {sdg}: {score}")

WorldHealthOrganization/godata - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.574
Zenysis/Harmony - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 1.0
unicef/hope - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.546
hotosm/tasking-manager - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.304
Rural-Environmental-Registry/core - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.79
ushahidi/platform - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.899
upyog/UPYOG - SDG 6: Ensure availability and sustainable management of water and sanitation for all: 0.399
